# Assignment 25: Prompting & LangChain Chains
**Student:** Parth Dadhaniya  
**Course:** Generative AI Engineering

This notebook covers:
- Prompt templates and message prompt templates
- Structured output from LLMs using Pydantic
- Simple, conditional, and parallel chains in LangChain
- Runnables and LCEL (LangChain Expression Language)
- Full LCEL RAG pipeline with Chroma vector store

## 1. Setup and Environment
Initialize paths and load the language model.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import config
llm = config.get_llm()
print('Model ready:', llm._llm_type)

## 2. Document Loading & Vector Store
Load text, CSV, and PDF documents from `data/` and index into Chroma.

In [ ]:
import vector_store
retriever = vector_store.get_retriever(k=2)
docs = retriever.invoke('What is machine learning?')
print(f'Retrieved {len(docs)} chunks.')
for i, d in enumerate(docs, 1):
    print(f'\nChunk {i} [{d.metadata.get("source")}]:\n{d.page_content[:100]}...')

## 3. Part 1 — Prompt Templates
### Task 1: PromptTemplate

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = """You are an AI tutor helping students learn Generative AI.
Answer the following question clearly and concisely in 2 sentences.

Question: {question}
Answer:"""

prompt = PromptTemplate(input_variables=['question'], template=template)
chain = prompt | llm | StrOutputParser()

for q in ['What is text vectorization in natural language processing?', 'Why do LLMs sometimes hallucinate facts?']:
    print('Q:', q)
    print('A:', chain.invoke({'question': q}).strip(), '\n')

### Task 2: ChatPromptTemplate & Message Templates

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful Python and Generative AI mentor.'),
    ('human', 'Hi, what does a vector store do?'),
    ('ai', 'A vector store indexes embeddings to enable fast similarity search across documents.'),
    ('human', '{user_input}')
])

chat_chain = chat_prompt | llm | StrOutputParser()
messages = chat_prompt.format_messages(user_input='How does FAISS compare to ChromaDB for storing embeddings?')
for m in messages:
    print(f'[{m.type.upper()}]: {m.content}')

print('\nModel output:\n', chat_chain.invoke({'user_input': 'How does FAISS compare to ChromaDB for storing embeddings?'}).strip())

## 4. Part 2 — Structured Output using Pydantic
### Task 3: Pydantic Output Schema (`Answer` model)

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

class Answer(BaseModel):
    answer: str = Field(description='Direct answer')
    confidence: float = Field(description='Confidence score between 0.0 and 1.0')
    source: str = Field(description='Source citation')

parser = PydanticOutputParser(pydantic_object=Answer)
schema_prompt = PromptTemplate(
    template='Answer the question with confidence and citation.\n{format_instructions}\nQuestion: {question}',
    input_variables=['question'],
    partial_variables={'format_instructions': parser.get_format_instructions()}
)

pydantic_chain = schema_prompt | llm | parser
ans_obj = pydantic_chain.invoke({'question': 'What is machine learning and how does it learn from data?'})
print('Parsed Answer:', ans_obj.answer)
print('Confidence   :', ans_obj.confidence)
print('Source       :', ans_obj.source)

### Task 4: Validation & Fallback

In [ ]:
from langchain_core.exceptions import OutputParserException

def parse_with_fallback(raw_text):
    try:
        return parser.parse(raw_text)
    except OutputParserException:
        print('[Notice] Invalid schema format. Applying fallback extraction.')
        clean = raw_text.replace('```json', '').replace('```', '').strip()
        return Answer(answer=clean, confidence=0.50, source='fallback_recovery')

malformed = 'This output is plain text without JSON.'
print(parse_with_fallback(malformed).model_dump())

## 5. Part 3 — Chains in LangChain
### Task 5: Simple Chain

In [ ]:
simple_prompt = PromptTemplate.from_template('Explain the role of {topic} in modern AI application development in 2 sentences.')
simple_chain = simple_prompt | llm | StrOutputParser()
print(simple_chain.invoke({'topic': 'Vector Embeddings'}).strip())

### Task 6: Conditional Chain (RunnableBranch)

In [ ]:
from langchain_core.runnables import RunnableBranch

def format_docs(docs):
    return '\n'.join(d.page_content for d in docs)

rag_prompt = PromptTemplate.from_template('Context: {context}\nQuestion: {question}\nAnswer:')
rag_branch = ({'context': (lambda x: x['question']) | retriever | format_docs, 'question': lambda x: x['question']} | rag_prompt | llm | StrOutputParser())

direct_prompt = PromptTemplate.from_template('Respond politely: {question}')
direct_branch = direct_prompt | llm | StrOutputParser()

def is_factual(x):
    return any(w in x['question'].lower() for w in ['policy', 'leave', 'csv', 'handbook', 'employee'])

conditional_chain = RunnableBranch((is_factual, rag_branch), direct_branch)

q_fact = {'question': 'What is the company leave policy according to the handbook?'}
q_chat = {'question': 'Hello, good morning! How are you doing today?'}

print('Factual Query Output :\n', conditional_chain.invoke(q_fact).strip())
print('\nConversational Output:\n', conditional_chain.invoke(q_chat).strip())

### Task 7: Parallel Chain (RunnableParallel)

In [ ]:
from langchain_core.runnables import RunnableParallel

ans_c = PromptTemplate.from_template('Provide a concise explanation of: {topic}') | llm | StrOutputParser()
sum_c = PromptTemplate.from_template('Summarize the main idea of {topic} in one short sentence:') | llm | StrOutputParser()
fol_c = PromptTemplate.from_template('Suggest 2 follow-up questions to study about {topic}:') | llm | StrOutputParser()

parallel_chain = RunnableParallel({
    'answer': ans_c,
    'summary': sum_c,
    'follow_up': fol_c
})

p_out = parallel_chain.invoke({'topic': 'Generative AI and Self-Attention Mechanisms'})
print('Summary   :', p_out['summary'].strip())
print('Follow-ups:\n', p_out['follow_up'].strip())

## 6. Part 4 — Runnables & LCEL
### Task 8: Runnables Basics & Passthrough

In [ ]:
from langchain_core.runnables import RunnablePassthrough

lcel_pipe = {'topic': RunnablePassthrough()} | PromptTemplate.from_template('Explain {topic} in simple terms.') | llm | StrOutputParser()
print(lcel_pipe.invoke('Machine Learning').strip())

### Task 9: LCEL RAG Pipeline (`Retriever | Prompt | LLM | Output Parser`)

In [ ]:
rag_pipe = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | PromptTemplate.from_template('Context:\n{context}\n\nQuestion: {question}\nAnswer:')
    | llm
    | StrOutputParser()
)

print('RAG Output:\n', rag_pipe.invoke('What employee departments and data fields are recorded in the CSV?').strip())

## 7. Part 5 — Observations & Insights
### Task 10: Observations & Insights

**1. Why structured output is important:**  
LLMs return unstructured text by default. Enforcing schemas with Pydantic ensures data fields (strings, numbers, lists) follow a strict contract so frontend apps, databases, and APIs can consume them without parsing errors.

**2. Advantages of LCEL over traditional chains:**  
LCEL uses the standard Unix-style pipe (`|`) syntax to connect components. It natively supports streaming, async calls, batching, and tracing without manual boilerplate.

**3. When to use parallel vs conditional chains:**  
- **Parallel Chains:** Use when multiple independent tasks can run at the same time on the same input (such as generating an answer and a summary simultaneously) to cut latency.
- **Conditional Chains:** Use when the execution path should change based on the user's input (such as routing factual questions to a vector database and conversational greetings to a direct model response).